# 爬虫全链路解剖
## 从一张网页，到数据库里的一行

**这份笔记本不花一分钱。** 它只读你已有的数据，不调用任何 AI。

---

### 它回答一个问题

> Seek 上一个岗位广告，怎么变成 FlowGT 数据库里的一行、
> 最后出现在会员的邮箱里？中间被动了哪些手脚？

### 七个阶段 · seven stages

```
  ①  决定抓哪些网址        URL planning        config.py + scraper.py
      ↓
  ②  真浏览器打开页面      headless browser    scraper.py  (Playwright)
      ↓  ← 拿到渲染完的 HTML
  ③  从 HTML 抠出字段      extraction          scraper.py  extract_job_cards
      ↓  ← 一个 dict，全是【原文】
  ④  存进本地 SQLite       local staging       scraper.py  save_job → jobs.db
      ↓
  ⑤  打包推给 API          push                push.py  → HTTP POST
      ↓  ← JSON，这里第一次把 '10h ago' 变成时间戳
  ⑥  服务端归一化 + 落库   ingest + normalise  functions/api/ingest/jobs.js
      ↓  ← 加上 region / role_family / seniority 这些【判断】
  ⑦  生产 D1 的一行        the fact row        migrations/0005-job-market.sql
```

### 主线例子 · the worked example

整份笔记本追同一个真实岗位：**Senior Software Engineer @ Seequent ULC**，
看它在每一站变成什么样子。

### ⚠️ 这份笔记本会告诉你四个已证实的问题（2026-08-12 全部修好）

它们不是理论风险，是我拿你库里的真数据数出来的。走到对应的 audit 格你会亲眼看到。

**修好了为什么还留着？** 因为四个问题共享同一个形状，而形状比修法值钱：
选择器失效返回空字符串、函数没被调用字段一直空、解析失败返回 None、被 Cloudflare 拦住说「本页没有岗位」—— **每一个失败都被渲染成了一个看起来正常的结果。**
下一个坏掉的东西，多半也长这样。

---
# 术语对照 · glossary

| 中文 | English | 一句话 |
|---|---|---|
| 无头浏览器 | **headless browser** | 有完整浏览器内核但不显示窗口 |
| 渲染 | **render** | 浏览器执行 JavaScript，把数据画成页面 |
| 选择器 | **selector** | 「页面上哪个元素」的地址，如 `[data-testid="job-card"]` |
| 退路 | **fallback** | 首选选择器失效时的备用方案 |
| 原文字段 | **raw field** | 一字不改抄下来的，如 `title_raw` |
| 归一化 | **normalisation** | 把杂乱原文映射成固定值，如 `Christchurch, Canterbury` → `christchurch` |
| 分类法 | **taxonomy** | 归一化用的规则表，存在 `role_taxonomy` |
| 幂等 | **idempotent** | 同一件事做两次，结果和做一次一样 |
| 更新插入 | **upsert** | 有就更新、没有就插入 |
| 全量扫描 | **full sweep** | 这一轮看到了全集，所以没看到的可以判定下架 |
| 宽限期 | **grace period** | 一次没看到不算下架，避免抓漏就误杀 |
| 时间戳 | **timestamp / epoch** | 从 1970 年起的秒数，如 `1786398766` |
| 事实表 | **fact table** | 存原始事实的表，一行一个岗位 |

> **术语一律保留英文。** 你要教组员，而他们查资料时看到的是英文。

---
# 准备：一个会讲解自己的打印工具

> **这一格干什么 what** 定义 `step()` 和 `verdict()`。每次 audit 都会打印【现在第几步】、【在验什么】、【结论是什么】、以及【这个结论意味着什么】。  
> **输入 input** 无  
> **应该看到 expected** `工具就绪 · ready`

In [ ]:
import sqlite3, json, pathlib, re, collections, subprocess, textwrap

HUNTER = pathlib.Path('../flowgt-job-hunter')
SITE   = pathlib.Path('../flowgt-website')
assert HUNTER.exists() and SITE.exists(), '三个仓库要在同一级目录下'

TOTAL_STEPS = 7

def step(n, zh, en):
    """打印阶段抬头，让你随时知道走到哪了。"""
    bar = '█' * n + '░' * (TOTAL_STEPS - n)
    print(f'\n{"═"*74}')
    print(f'  第 {n} / {TOTAL_STEPS} 步   {bar}')
    print(f'  {zh}')
    print(f'  {en}')
    print('═'*74)

def audit(what, why):
    """声明这一格在【验证】什么，以及为什么值得验。"""
    print(f'\n🔍 AUDIT：{what}')
    print(f'   为什么要验：{why}\n')

def verdict(ok, finding, means):
    """结论 + 【它意味着什么】。只给数字不给解释，等于没说。"""
    mark = '✓' if ok else '✗'
    print(f'   {mark} 结论 finding：{finding}')
    print(f'   → 意味着 means：{means}')

def show(title, rows):
    print(f'   {title}')
    for k, v in rows: print(f'     {k:<18} {v}')

print('工具就绪 · ready')

> **这一格干什么 what** 锁定主线例子 —— 整份笔记本都追这一个岗位。  
> **输入 input** `../flowgt-job-hunter/jobs.db`  
> **应该看到 expected** 一个岗位的本地原始记录，13 个字段
  
> **看到别的 otherwise** 找不到就换一个：把 `EXAMPLE_ID` 改成库里任意一个 id

In [ ]:
con = sqlite3.connect(HUNTER / 'jobs.db')
con.row_factory = sqlite3.Row

EXAMPLE_ID = 'seek-93884197'   # 换成任意一个 id 都行 / any id works
row = con.execute('SELECT * FROM jobs WHERE id=?', (EXAMPLE_ID,)).fetchone()
if row is None:
    row = con.execute('SELECT * FROM jobs ORDER BY scraped_at DESC LIMIT 1').fetchone()
    EXAMPLE_ID = row['id']
    print(f'（原例子不在库里，改用 {EXAMPLE_ID}）')

EX = dict(row)
print(f'主线例子 the worked example：{EX["job_title"]} @ {EX["company"]}\n')
for k, v in EX.items():
    print(f'  {k:<13} {repr(v)[:74]}')

---
# ① 决定抓哪些网址 · URL planning

**代码在** `config.py` + `scraper.py:88`（关键词）和 `scraper.py:106`（分类页）

这一步不碰网络，只把 7 个常量变成一串具体网址。

## 两条路，主次分明

```
  路线 A（主）分类落地页 classification pages
     2 个分类 × 最多 8 页 = 最多 16 次页面加载
     https://nz.seek.com/jobs-in-information-communication-technology?daterange=1
     → 这一类的【全集】，直接翻页

  路线 B（补）关键词搜索 keyword search
     33 个关键词 × 3 个地区 × 最多 4 页 = 最多 396 次加载
     https://nz.seek.com/software-developer-jobs/in-Christchurch-Canterbury?daterange=1&classification=6281%2C6304
     → 捞被归到 ICT 之外的技术岗（数据科学家常在 Science & Technology）
```

**为什么分类页是主路：** 396 次加载 vs 16 次，而且关键词**会漏** ——
标题和正文里没有那 33 个词的岗位，关键词永远找不到，分类页却一定有。

**`daterange=1`** = 只要 24 小时内发布的。这个数字曾经是 7，
而邮件标题写着「今日岗位」—— 那就是 2026-08-09 那次事故。

> **这一格干什么 what** 验证 URL 真的是这么拼出来的 —— 直接调用源码里的函数。  
> **输入 input** `scraper.py` 的两个 URL 构造函数  
> **应该看到 expected** 四条真实 URL，以及两条路线的加载次数对比

In [ ]:
step(1, '决定抓哪些网址', 'URL planning — turning constants into concrete URLs')
import sys; sys.path.insert(0, str(HUNTER))
import importlib, config, scraper
importlib.reload(config); importlib.reload(scraper)

audit('URL 是不是真的按说明那样拼出来的',
      '说明文档会过期，函数不会。直接调它，看它吐什么。')

print('   路线 A · 分类页 classification（主路 primary）')
for slug, why in config.CLASSIFICATION_PAGES:
    print(f'     {scraper.build_classification_url(slug, 1)}')
print(f'     第 3 页长这样：{scraper.build_classification_url(config.CLASSIFICATION_PAGES[0][0], 3)}')

print('\n   路线 B · 关键词 keyword（补充 supplementary）')
print(f'     {scraper.build_search_url(config.KEYWORD_GROUPS[0], config.LOCATIONS[0], 1)}')

a = len(config.CLASSIFICATION_PAGES) * config.MAX_PAGES_PER_CLASSIFICATION
b = len(config.KEYWORD_GROUPS) * len(config.LOCATIONS) * config.MAX_PAGES_PER_KEYWORD
verdict(True, f'路线 A 最多 {a} 次加载，路线 B 最多 {b} 次',
        f'B 比 A 贵 {b//a} 倍，而且还会漏掉不含那 {len(config.KEYWORD_GROUPS)} 个关键词的岗位。'
        f'所以 A 是主路，B 只是补网。')

verdict(config.DATE_POSTED_DAYS == 1, f'daterange = {config.DATE_POSTED_DAYS}（天）',
        '只要 24 小时内的。这个数曾经是 7，而邮件标题写着「今日岗位」—— 2026-08-09 事故。')

---
# ② 真浏览器打开页面 · headless browser

**代码在** `scraper.py:251`

## 为什么非要浏览器不可

```
  纯 HTTP（requests.get）              真浏览器（Playwright + Chromium）
        ↓                                      ↓
  收到 HTML 空壳                        收到 HTML → 执行 JS → 卡片出现
        ↓                                      ↓
    0 张卡片                                32 张卡片
```

Seek 的岗位是 **JavaScript 渲染**的。服务器发回来的第一份 HTML 里没有岗位，
只有一段 JS。**没有浏览器内核 = 没有 JS 引擎 = 永远看到空页面。**

## 四件「像人」的事

| 代码 | 干什么 |
|---|---|
| `user_agent="…Chrome/124…"` | 自报家门说是 Mac 上的 Chrome |
| `viewport={1280, 800}` | 假装是正常尺寸的屏幕 |
| `human_delay(1.0, 2.0)` | 每页之间随机停 1–2 秒 |
| `scroll_page()` | 随机往下滚 2–4 次（有内容要滚动才加载）|

另有两行在抹掉「我是自动化程序」的标记（`--disable-blink-features=AutomationControlled`
和 `navigator.webdriver` 覆盖）。**这是既有代码,不是新加的** ——
而且今天 Seek 上了 Cloudflare 之后它们已经不管用了：Cloudflare 拦的是**机房 IP**，
不是浏览器指纹。

## 2026-08-11 补的那道守卫

`assert_real_page()`（`scraper.py:196`）—— **一张验证页不是「这一类没有岗位」**。
在它之前，101 个页面全部报「本页没有岗位」，工作流一路绿。

> **这一格干什么 what** 验证浏览器配置，以及那道新守卫真的存在并被调用。  
> **输入 input** `scraper.py` 源码  
> **应该看到 expected** 启动参数、四件「像人」的事、`assert_real_page` 的调用点

In [ ]:
step(2, '真浏览器打开页面', 'headless browser — because the jobs are rendered by JavaScript')
src = (HUNTER / 'scraper.py').read_text(encoding='utf-8')

audit('浏览器是不是真的 Chromium，还是伪装的 HTTP 请求',
      '这决定了它能不能看到 JS 渲染出来的岗位卡片。')

has_pw   = 'async_playwright' in src and 'chromium.launch' in src
has_http = bool(re.search(r'^import requests|requests\.get', src, re.M))
verdict(has_pw and not has_http,
        f'Playwright + Chromium = {has_pw}　纯 HTTP 抓取 = {has_http}',
        '真浏览器。所以它慢（每页要等 JS 跑完），但也所以它看得见岗位。')

print()
audit('那道「被盘问 ≠ 没岗位」的守卫在不在，而且被调用了吗',
      '2026-08-11：没有它的时候，101 张验证页被报成 101 次「本页没有岗位」。')

defined = 'async def assert_real_page' in src
called  = src.count('await assert_real_page(')
verdict(defined and called >= 2,
        f'定义 = {defined}　被调用 {called} 处',
        '两条路线（分类页和关键词页）各一处。少于 2 说明有一条路还是哑的。')

print()
print('   四件「像人」的事 / four politeness-and-camouflage measures:')
for name, pat in [('user_agent', 'Chrome/124'), ('viewport', 'viewport='),
                  ('随机延迟 delay', 'human_delay'), ('滚动 scroll', 'scroll_page')]:
    print(f'     {name:<18} {"✓" if pat in src else "✗"}')

---
# ③ 从 HTML 抠出字段 · extraction

**代码在** `scraper.py:218` `extract_job_cards()`

## 怎么抠

```
  找到所有卡片   [data-testid="job-card"]        ← 入口选择器，它死了就全盘归零
        ↓  对每一张卡片：
  标题   a[data-automation="jobTitle"] → 退路 job-card-title, h3 a  ← 08-12 换的主选择器
  公司   [data-testid="company-name"] , [data-automation="jobCompany"]
  地点   [data-testid="job-location"] , [data-automation="jobLocation"]
  工时   读屏文字 "This is a Full time job"     ← 卡片上【没有】对应属性
  简介   [data-automation="jobShortDescription"] ← 08-12 新增，免费的 ~250 字
  薪资   [data-testid="job-salary"]   , [data-automation="jobSalary"]
  年龄   [data-automation="jobListingDate"]      ← "10h ago" 这种字符串
  分类   [data-automation="jobClassification"] / jobSubClassification
```

**每个字段都有退路（fallback）。** 因为 Seek 改过版 —— 一个匹配不到的选择器
**不会报错，只会安静地返回空字符串**。

## ⚠️ 下面的 audit 会告诉你两件事

这一段的说明写得很好，但**现实和它对不上**。自己看数字。

> **这一格干什么 what** 验证每个选择器在**今天的 Seek 上**还活着吗。  
> **输入 input** `audit-local/audit.json`（`flowgt-job-hunter/audit_seek.py` 的产物）+ `jobs.db`  
> **应该看到 expected** 一张选择器存活表，以及每个字段在 155 行里的填充率
  
> **看到别的 otherwise** 没有 audit-local 就先去 flowgt-job-hunter 跑 `python3 audit_seek.py --out audit-local`

In [ ]:
step(3, '从 HTML 抠出字段', 'extraction — selectors, and the silence when they miss')

audit('选择器在今天的 Seek 上还活着吗',
      '一个匹配不到的选择器【不报错】，只返回空字符串。所以只能靠数数发现它死了。')

ap = HUNTER / 'audit-local' / 'audit.json'
if ap.exists():
    a = json.load(open(ap))
    print(f"   {'选择器 selector':<34} {'ICT':>5} {'Sci':>5} {'关键词':>6}")
    keys = ['[data-testid="job-card"]', 'a[data-automation="jobTitle"]',
            'a[data-testid="job-title"]', 'h3 a, h2 a', '[data-automation="jobListingDate"]']
    for k in keys:
        vals = [p.get('selectors', {}).get(k, '?') for p in a['pages']]
        print(f"   {k:<34} " + ''.join(f'{v:>6}' for v in vals))
    dead = a['pages'][0]['selectors'].get('a[data-testid="job-title"]', 1) == 0
    verdict(not dead,
            '主标题选择器 a[data-testid="job-title"] 命中 0 个' if dead else '主选择器还活着',
            '整个抓取一直靠退路 h3 a 在跑。它没坏，但它在【裸奔】——'
            '哪天退路也变了，就会像 08-11 那样安静地归零。' if dead else '正常。')
else:
    print('   （没有 audit-local。先去 flowgt-job-hunter 跑 python3 audit_seek.py --out audit-local）')

print()
audit('每个字段实际填充率是多少',
      '选择器失效不会报错。填充率是唯一能发现「这个字段一直是空的」的办法。')
tot, = con.execute('SELECT COUNT(*) FROM jobs').fetchone()
for col in ('job_title', 'company', 'location', 'job_type', 'salary', 'posted_date', 'jd', 'job_class'):
    n, = con.execute(f"SELECT COUNT(*) FROM jobs WHERE {col} IS NOT NULL AND {col}!=''").fetchone()
    pct = 100*n/tot
    flag = '  ← ⚠️ 从来没有值' if n == 0 else ''
    print(f'     {col:<13} {n:>4}/{tot}  {pct:5.1f}%{flag}')

jt, = con.execute("SELECT COUNT(*) FROM jobs WHERE job_type IS NOT NULL AND job_type!=''").fetchone()
jd, = con.execute("SELECT COUNT(*) FROM jobs WHERE jd IS NOT NULL AND jd!=''").fetchone()
# 注意看【两栏】：全库会被历史行稀释，一个今天刚死的选择器只在「最近 24h」那栏现形。
rec, = con.execute("SELECT COUNT(*) FROM jobs WHERE scraped_at > datetime('now','-24 hours')").fetchone()
jt, = con.execute("SELECT COUNT(*) FROM jobs WHERE job_type IS NOT NULL AND job_type!=''").fetchone()
jd, = con.execute("SELECT COUNT(*) FROM jobs WHERE jd IS NOT NULL AND jd!=''").fetchone()
verdict(jt > 0, f'job_type {jt}/{tot} · jd {jd}/{tot} · 最近 24h 共 {rec} 行',
        'job_type 修好之前是 0/155：两个选择器在 Seek 上根本不存在，而没有任何地方报错。'
        ' jd 为 0 是【正常】的 —— 日常采集不抓正文，要 run.py --scrape --with-jd 才抓。'
        ' 旧行不会自己变：它们是坏代码抓的，靠下一次重抓的自愈补空格子。')

---
# ④ 存进本地 SQLite · local staging

**代码在** `scraper.py:28` `init_db()` · `scraper.py:48` `job_id()` · `scraper.py:69` `save_job()`

## 身份问题：为什么不能用整条 URL 做 id

Seek 的结果链接长这样：

```
https://www.seek.co.nz/job/93884197?type=standard&ref=search-standalone&origin=cardTitle…
                            ▲▲▲▲▲▲▲▲ 这才是身份   ▲ 问号后面是每次搜索都不同的追踪码
```

对整条 URL 取 md5，同一个岗位在 33 个关键词下会得到 **33 个不同的 id**。
实际发生过：岗位 93884197 存了四份，**844 行里其实只有 244 个真岗位**。

所以 `job_id()` 用正则抠出岗位号：`/job/(\d+)` → `seek-93884197`。

## `INSERT OR IGNORE` = 幂等 idempotent

同一个岗位抓到第二次，什么都不做。所以重复跑不会产生重复行。

> **这一格干什么 what** 验证 id 的稳定性 —— 同一个岗位在不同搜索下会不会得到同一个 id。  
> **输入 input** `scraper.py` 的 `job_id()`  
> **应该看到 expected** 同一个岗位号、三条不同追踪参数的 URL → 同一个 id

In [ ]:
step(4, '存进本地 SQLite', 'local staging — identity, and why the URL cannot be it')

audit('同一个岗位在不同搜索下会不会被当成不同的岗位',
      '曾经会。844 行里只有 244 个真岗位 —— 因为 id 是对整条带追踪码的 URL 取哈希。')

same = ['https://www.seek.co.nz/job/93884197?type=standard&ref=search-standalone',
        'https://www.seek.co.nz/job/93884197?type=promoted&ref=keyword&origin=cardTitle',
        'https://www.seek.co.nz/job/93884197']
ids = [scraper.job_id(u) for u in same]
for u, i in zip(same, ids): print(f'     {i}   ←  {u[:62]}…')
verdict(len(set(ids)) == 1, f'三条 URL → {len(set(ids))} 个不同 id',
        '身份靠岗位号，不靠 URL。所以同一个岗位被 33 个关键词各找到一次，仍然只有一行。')

print()
audit('本地库现在有没有重复', '幂等失效的表现就是行数虚高。')
tot, = con.execute('SELECT COUNT(*) FROM jobs').fetchone()
uniq, = con.execute('SELECT COUNT(DISTINCT id) FROM jobs').fetchone()
verdict(tot == uniq, f'{tot} 行 / {uniq} 个不重复 id',
        '一致 = INSERT OR IGNORE 在生效，重复跑不会灌水。' if tot == uniq else '不一致 = 有重复，去查 job_id()。')

---
# ⑤ 打包推给 API · push

**代码在** `push.py`

## 这一步做的最重要的一件事：把「10h ago」变成时间戳

```
  本地           posted_date = '10h ago'      ← 一个字符串，人读的
      ↓  parse_age()
  推送           postedAt    = 1786398766     ← 时间戳，机器能比较的
```

**为什么这一步致命：** 邮件的筛选条件是「36 小时内发布的」。
没有时间戳就没法比较 —— 而 `parse_age()` **解析不出来时返回 `None`**，不是返回「现在」。

这是 2026-08-09 事故的修法：以前解析不出来就当成「刚刚」，
于是一周前的岗位被当成今天的发出去，会员点进 Seek 看到「Posted 6d ago」。

> **宁可安静地少一条，也不要自信地错一条。**

## ⚠️ 但这个修法有代价，下面 audit 会算给你看

> **这一格干什么 what** 验证时间解析，并**数出代价** —— 有多少岗位因此永远发不出去。  
> **输入 input** `push.py` 的 `parse_age()` + `jobs.db` 里全部 `posted_date` 值  
> **应该看到 expected** 解析示例，然后是解析失败的行数和它们的下场

In [ ]:
step(5, '打包推给 API', 'push — where a human-readable age becomes a machine timestamp')
import push; importlib.reload(push)
from datetime import datetime, timezone

audit('「10h ago」这种字符串怎么变成时间戳',
      '邮件要按「36 小时内」筛选，字符串没法比较大小。')

now = datetime.now(timezone.utc)
for s in ['10h ago', '2d ago', '45m ago', 'Featured', 'New to you', '']:
    got = push.parse_age(s, now) if hasattr(push, 'parse_age') else None
    show_v = datetime.fromtimestamp(got, timezone.utc).strftime('%m-%d %H:%M') if got else 'None ← 解析不出来'
    print(f'     {s!r:<16} → {show_v}')

print()
audit('解析不出来的有多少，它们会怎么样',
      '这是那个修法的【代价】。修法本身是对的，但代价要说出来。')
vals = collections.Counter(r[0] for r in con.execute('SELECT posted_date FROM jobs'))
bad = {k: v for k, v in vals.items() if not re.match(r'^\d+\s*[mhd]', str(k or ''))}
n_bad = sum(bad.values())
print(f'     {sum(vals.values())} 行，{len(vals)} 种不同的值')
print(f'     解析不出年龄的：{bad}')
verdict(n_bad == 0, f'{n_bad} 行的 postedAt 会是 null',
        f'send-digest.js 里有一条硬条件 `posted_at IS NOT NULL`，'
        f'所以这 {n_bad} 个岗位【永远不会】出现在任何一封邮件里。'
        f'「Featured」是 Seek 的推广位，卡片上的日期格子原文就是 "Featured"。'
        f'08-12 之前我们停在「不知道就不发」——【对了一半】：不该编日期是对的，'
        f'把「不知道」当终点是错的。现在会去详情页问真日期（Featured → 13h ago）。')

---
# ⑥ 服务端归一化 + 落库 · ingest & normalise

**代码在** `flowgt-website/functions/api/ingest/jobs.js`

## 这一步给数据**加判断**

```
  收到（全是原文 raw）                  存下（原文 + 判断）
  title    Senior Software Engineer  →  title_raw    Senior Software Engineer   ← 一字不改
                                        role_family  software      ← 判断出来的
                                        role_group   dev           ← 判断
                                        seniority    senior        ← 判断
  location Christchurch, Canterbury  →  location_raw Christchurch, Canterbury   ← 原文
                                        region       christchurch  ← 归一化
```

**原文永远保留。** 归一化规则改了，可以拿原文重跑；原文丢了就永远回不来。
这就是为什么表里同时有 `title_raw` 和 `role_family`。

## 一道闸门：这是不是 IT 岗位

`isITRole()` **信 Seek 自己的分类，不靠标题猜**。

2026-08-10 实测：用标题猜的那一版，把 79 个来自 ICT 分类页的岗位**拒掉了 34 个** ——
包括 `Digital Graduate - Data and AI`、`ICT Service Desk Analyst`。

> **来源已经回答过的问题，不要用启发式再答一遍。**

> **这一格干什么 what** 验证归一化真的发生了 —— 拿生产库里的真实行看。  
> **输入 input** 生产 D1（只读查询）  
> **应该看到 expected** 同一个岗位的原文字段 vs 判断字段并排
  
> **看到别的 otherwise** 查不到就是没登录 Cloudflare：`cd ../flowgt-website && npx wrangler login`

In [ ]:
step(6, '服务端归一化 + 落库', 'ingest & normalise — where raw text gains judgement')

audit('归一化到底加了什么',
      '原文和判断必须能分清。分不清的话，规则改了就没法重跑。')

q = ("SELECT external_id, title_raw, role_family, role_group, seniority, "
     "location_raw, region, posted_at, first_seen_day, last_seen_day, is_open "
     f"FROM jobs WHERE external_id='{EXAMPLE_ID}' LIMIT 1;")
r = subprocess.run(['npx','wrangler','d1','execute','flowgt','--remote','--command',q],
                   cwd=SITE, capture_output=True, text=True,
                   env={**__import__('os').environ, 'CI':'1', 'WRANGLER_SEND_METRICS':'false'})
m = re.search(r'"results":\s*\[(.*?)\]\s*,\s*"success"', r.stdout, re.S)
rows = json.loads('['+m.group(1)+']') if m else []

if rows:
    p = rows[0]
    print('     原文 RAW（一字不改 / never altered）')
    for k in ('external_id','title_raw','location_raw'): print(f'       {k:<15} {p[k]}')
    print('     判断 DERIVED（系统加的 / added by the system）')
    for k in ('region','role_family','role_group','seniority'): print(f'       {k:<15} {p[k]}')
    print('     存活 LIFECYCLE')
    for k in ('posted_at','first_seen_day','last_seen_day','is_open'): print(f'       {k:<15} {p[k]}')
    verdict(True, f"本地 posted_date='{EX['posted_date']}' → 生产 posted_at={p['posted_at']}",
            '左边是人读的字符串，右边是机器能比较的时间戳。这一步就是整条链路的分水岭。')
else:
    print('     （生产库里没有这一条，或者没登录 Cloudflare）')

---
# ⑦ 生产 D1 的一行 · the fact row

**代码在** `flowgt-website/migrations/0005-job-market.sql:32`

## 三组字段，三种性质

| 组 | 字段 | 性质 |
|---|---|---|
| **原文 raw** | `title_raw` `company` `location_raw` `url` | 抄来的，永不修改 |
| **判断 derived** | `role_family` `role_group` `seniority` `region` `taxonomy_ver` | 算出来的，规则变了可重算 |
| **存活 lifecycle** | `first_seen_day` `last_seen_day` `is_open` `posted_at` | 这个岗位的一生 |

## `first_seen_day` 和 `posted_at` 不是一回事

```
  posted_at        岗位【发布】的时刻       ← Seek 说的
  first_seen_day   我们【第一次看见】的日子  ← 我们说的
```

**2026-08-09 事故就是把这两个搞混了。** 第一次全量采集把 Seek 上过去一周的岗位
一次性入库，它们的 `first_seen_day` 全是当天，于是全部被当成「今日岗位」发出去。

## 薪资列有个警告

新西兰大多数岗位不写薪资，所以这几列大部分是空的。
**报表必须同时给出「有薪资的样本量」** —— 否则中位数会骗人。

> **这一格干什么 what** 看真实的表结构和填充率。  
> **输入 input** 生产 D1  
> **应该看到 expected** jobs 表的列数、各组字段的填充率、以及薪资样本量

In [ ]:
step(7, '生产 D1 的一行', 'the fact row — raw, derived, lifecycle')

audit('生产库里这些字段实际填充成什么样',
      '表结构说「可以有」，填充率说「实际有没有」。差别很大。')

q2 = ('''SELECT COUNT(*) AS total,
  SUM(posted_at IS NOT NULL) AS has_posted,
  SUM(role_family IS NOT NULL) AS has_family,
  SUM(role_group IS NOT NULL) AS has_group,
  SUM(seniority IS NOT NULL) AS has_seniority,
  SUM(salary_min IS NOT NULL) AS has_salary,
  SUM(is_open=1) AS still_open
FROM jobs;''')
r2 = subprocess.run(['npx','wrangler','d1','execute','flowgt','--remote','--command',q2],
                    cwd=SITE, capture_output=True, text=True,
                    env={**__import__('os').environ, 'CI':'1', 'WRANGLER_SEND_METRICS':'false'})
m2 = re.search(r'"results":\s*\[(.*?)\]\s*,\s*"success"', r2.stdout, re.S)
if m2:
    s = json.loads('['+m2.group(1)+']')[0]
    t = s['total']
    for k, label in [('has_posted','有发布时间 posted_at'), ('has_family','有方向 role_family'),
                     ('has_group','有分组 role_group'), ('has_seniority','有级别 seniority'),
                     ('has_salary','有薪资 salary_min'), ('still_open','还挂着 is_open')]:
        print(f'     {label:<28} {s[k]:>5}/{t}  {100*s[k]/t:5.1f}%')
    verdict(s['has_salary'] < t*0.5,
            f"薪资样本量 {s['has_salary']}/{t}（{100*s['has_salary']/t:.0f}%）",
            '新西兰大多数岗位不写薪资。所以任何薪资统计都必须同时给出样本量 —— '
            '拿 30% 的样本算中位数然后说成「市场行情」，那是在骗自己。')

---
# 落库之前的闸门 · the gate before the database

上面七个阶段讲的是**数据怎么流**。这一段讲的是**放行之前查什么**。

## 为什么在【推之前】验，而不是推完再看

推完再看，坏数据已经在生产库里了 —— **而生产库直接喂会员的邮箱**。

2026-08-10 那天就是这样：4 个岗位进了库，17 个会员收到一封几乎空的信，
而所有的绿灯都还亮着。

## 七道检查 · seven checks

| # | 在验什么 | FAIL 的意思 |
|---|---|---|
| 0 | 数据源 raw source | 库是空的 —— 推上去等于什么都没发生 |
| 1 | 字段完整性 field completeness | 关键字段缺了，那一行下游用不了 |
| 2 | 身份唯一 identity | 同一岗位存成多行，数字会虚高 |
| 3 | 时间可解析 parseable dates | 解析不出 = 永远发不出去 |
| 4 | 分类可信 source classification | 没分类就要靠标题猜，而猜会拒错 |
| 5 | 和源头对账 reconcile | **唯一向外的检查** —— 跟自己比永远是对的 |
| 6 | 推送预演 dry run | 会推 0 条，那就别推 |

## 三个等级 · three levels

```
  PASS   放行
  WARN   可以推，但你要知道这件事（比如「Featured 岗位永远发不出去」）
  FAIL   别推。推上去会让生产库变差，而生产库直接喂邮箱。
```

### ⚠️ 2026-08-12 的教训：这个 audit 自己误报过一次

第一版的第 5 步，拿**昨天**的 Seek 数字去比**正在跑还没跑完**的采集，
报出 `15% → FAIL`。数据其实没问题。

> **一个会误报的闸门，比没有闸门更糟。**
> 它会训练操作它的人直接点过去 —— 于是下一次真的出问题时，也没人看。

修法：采集在跑 / 基准超过 6 小时 / 根本没有基准 —— 三种情况都**不许判 FAIL**，
只说「现在没法判断」。

> **这一格干什么 what** 跑完整的落库前 audit。**它是一个常驻脚本，这里只是调用它** —— 逻辑只有一份，笔记本和命令行跑的是同一套。  
> **输入 input** `../flowgt-job-hunter/jobs.db` + `audit-local/audit.json` + 一次 dry-run  
> **应该看到 expected** 七段检查，每段都有【在验什么 / 实测数字 / PASS·WARN·FAIL / 这意味着什么】
  
> **看到别的 otherwise** 有 FAIL 就不要推 —— 脚本最后会告诉你原因

In [ ]:
import subprocess, sys

# ★ 刻意【调用】而不是【复制】那个脚本。
#   复制一份逻辑到笔记本里，两边迟早会分叉，而分叉的表现是
#   「命令行说可以推、笔记本说不能推」—— 那时你不知道该信哪个。
# ★ Deliberately calling the script rather than reimplementing it: a second
#   copy drifts, and the symptom is the terminal and the notebook disagreeing
#   about whether it is safe to push.
r = subprocess.run([sys.executable, str(HUNTER / 'bin' / 'audit-before-push')],
                   cwd=HUNTER, capture_output=True, text=True, timeout=300)
print(r.stdout[-6000:])
if r.stderr.strip(): print('STDERR:', r.stderr[-800:])

## 读懂它的输出 · reading the result

最后一行是结论。三种可能：

```
  ✓ 可以推                        → 跑推送命令
  ✓ 可以推（N 条 WARN 不挡路）     → 也可以推，但先读一遍那几条 WARN
  ✗ 有 N 条 FAIL —— 不要推        → 先修，修完再跑一次这个 audit
```

**WARN 不是「没事」，是「已知，且你应该知道」。** 现在的两条常驻 WARN：

| WARN | 为什么留着不修 |
|---|---|
| `jd` 为空 | **正常**。日常采集不抓正文（每个岗位要多开一页）。要正文跑 `--with-jd` |
| 旧行的 `job_type` / `teaser` 为空 | 它们是 08-12 之前坏代码抓的。重抓会自愈，但不会自己变 |

08-12 之前这里还有两条，现在**不该**再出现 —— 出现了就是回归：
`job_type` 全库为空、`Featured` 岗位解析不出时间。

---
# 全链路总账 · the whole path in one table

| 阶段 | 输入 | 输出 | 关键动作 |
|---|---|---|---|
| ① URL | 7 个常量 | 最多 412 条网址 | 分类页为主，关键词补网 |
| ② 浏览器 | 网址 | 渲染完的 HTML | 必须真浏览器，因为岗位是 JS 画的 |
| ③ 抠字段 | HTML | dict（全是原文）| 每个字段都有 fallback |
| ④ 本地库 | dict | `jobs.db` 一行 | 身份用岗位号，不用 URL |
| ⑤ 推送 | 本地行 | JSON | **`'10h ago'` → 时间戳** |
| ⑥ 归一化 | JSON | D1 一行 | 原文 + 判断，原文永不改 |
| ⑦ 事实行 | — | 生产 D1 | 三组字段：raw / derived / lifecycle |

---

# ⚠️ 这条链路上四个已证实的问题 —— 2026-08-12 全部修好

跑完上面的 audit 你已经亲眼看到它们了。四个都留在这里，**因为它们比修法值钱**：
它们是同一个形状的四个样本，而那个形状还会再来。

### 1. `fetch_jd()` 是死代码 —— JD 正文一个都没有
`scraper.py:291` 定义了它，**全仓库没有一处调用**。所以 `jd` 字段 0/155。
而 `matcher.py:43` 的闸门正好是 `WHERE jd != ''` —— 于是 **Claude 打分那一步
从来没有评过一份岗位**，却每次都干干净净地跑完、报告里一条不显示，
看起来像「今天没有匹配的」。

> **✅ 修法（08-12）：** 改名 `fetch_detail()`，在 `scrape()` 的第三路真的被调用，
> 走 `--with-jd` 开关。实测拿回 1521 / 2294 / 3012 / 3445 / 4737 字。
> 顺带发现旧函数里那两个选择器在详情页上**也是 0 命中** —— 就算它当年被调用了，
> 也只会掉到 `main` 兜底，把 5000 字页面家具当成职位描述存进去。

### 2. 主标题选择器已经死在 Seek 上
`a[data-testid="job-title"]` 在今天的 Seek 上命中 **0 个**（三个页面都是）。
整个抓取靠退路 `h3 a` 在跑。**它没坏，但它在裸奔** —— 退路哪天也变，就会像
08-11 那样安静地归零。

> **✅ 修法（08-12）：** 主选择器换成 `a[data-automation="jobTitle"]`，
> 对着 57 张存档卡片实测 **57/57**。真正的修法不是换选择器，是
> `test_selectors.py` —— 一道「哪个选择器死了就会响」的闸门。

### 3. `job_type` 永远是空的
两个选择器在 Seek 上都不存在，0/155。**没有任何地方报错**，因为
「选择器匹配不到」返回的是空字符串，不是异常。

> **✅ 修法（08-12）：** 卡片上确实没有这个属性 —— 存档页里唯一带 `WorkType` 的
> 是**筛选面板**（`toggleWorkTypeButton`、`refineWorkType`，各 1 次），那是页面家具。
> 真正的位置是给读屏软件的一句话 `This is a Full time job`，57/57 张卡都有。
> 用 `text_content()` 而不是 `inner_text()`：那句话视觉隐藏，`inner_text` 只返回
> 渲染出来的文字。**代价说清楚**：这是无障碍文案，不是数据契约，Seek 改文案它就没了 ——
> 所以闸门里有一条断言守着它。

### 4. 「Featured」岗位永远进不了邮件
Seek 的推广位不显示年龄 → `parse_age` 返回 `None` → `posted_at` 为 null →
被 `send-digest.js` 的硬条件挡住，约占 6%。

> **✅ 修法（08-12）：** 之前写的「这是对的设计」**只对了一半**。
> 对的部分是：**不该编一个日期填进去**，那是撒谎。
> 错的部分是：把「不知道」当成了终点 —— 其实可以**去问**。
> 详情页上有 `Posted 13h ago`，实测置顶卡 `'Featured'` → `'13h ago'` / `'16h ago'`。
> 日期必须锚在 `Posted` 上：页面右边「更多岗位」里也有 `2d ago`、`6d ago`，
> 不锚就会把别人的日期安到这个岗位头上 —— 那种错不会报，只会静静地错。
> 交叉验证：普通卡片上写 `20h ago`，详情页也回 `20h ago`，一致。

### 5.（顺带挖出来的）修好的字段补不到旧行上
`INSERT OR IGNORE` 遇到已存在的行**直接跳过**，所以用旧代码抓进来的行
**永远补不上后来才修好的字段**。08-12 当天就撞上：早上 171 行用坏掉的选择器抓的，
重跑采集也不会变，因为它们「已经存在」。

> **✅ 修法：** 重抓时填**现在是空的**格子，绝不覆盖已有的值。
> `posted_date` 刻意排除 —— 它存的是相对时间（`3h ago`），要配着当初的 `scraped_at`
> 才有意义；拿今天的 `3h ago` 覆盖三天前那一行，会把三天前的岗位说成三小时前。

---

### 一个贯穿始终的形状

上面四个问题，**没有一个会报错**。

| | 表现 |
|---|---|
| 选择器失效 | 返回空字符串 |
| 函数没被调用 | 字段一直是空 |
| 解析失败 | 返回 None |
| 被 Cloudflare 拦住 | 「本页没有岗位」|

> **每一个失败都被渲染成了一个看起来正常的结果。**
> 这就是为什么这条链路上每一处都要有 audit，而不是等着它报错。